# TN-VQE on IBM hardware: generating and costing the campaign

This notebook builds the stage-0 simulator screen and the stage-1 hardware
campaign, costs every hardware run against measured IBM billing, and
partitions the result into batches ordered by what each one proves. It is
the executable counterpart to
[the campaign README](README.md),
which carries the experimental reasoning behind the choices made here.

A **run** is one VQE or TN-VQE optimisation of one Hamiltonian by one
method, with its own ansatz, mapper, measurement method and evaluation
budget. One run is one row of the generated CSV.

Everything below needs only the standard library and this repository. No
IBM credentials are used, and nothing is submitted to hardware.

In [ ]:
import pathlib
import sys

REPO = pathlib.Path.cwd()
while not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent

# The campaign machinery is not an installed package, so it is imported
# from utils/ by path. That has to happen before the next cell's imports.
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "utils"))

In [ ]:
import build_benchmark_matrix as matrix
import split_benchmark_batches as batches

CAMPAIGN = REPO / "data" / "benchmarks" / "ibm_tn-vqe_qesem"
print(f"repository: {REPO}")

## Stage 0: the simulator screen

Stage 0 carries the campaign's breadth, and buys no QPU time. Six factors
fully crossed: the closed 2x2x2 of chemistry cells in `SCREENED`, four
ansatz families, three optimizers, three methods, two measurement methods
and two simulator backends. The one hole is UCCSD on mol_map, which has no
builder and no pinned circuit (`UCCSD_MAPPERS`).

It exists because the campaign cannot afford to guess. Measurement counts
carried as assumptions put an earlier revision 24x over budget, and the
counts used now are greedy upper bounds a real run beats by about 2x.

## Stage 1: the hardware screen

Stage 1 buys only what a simulator cannot answer: device error, and the
QPU time a method really consumes. So it takes one ansatz of the four, one
optimizer of the three, one measurement method per mapper, and the four
chemistry cells in `STAGE1_HARDWARE`. Everything it drops is run in stage 0
rather than abandoned.

In [ ]:
stage0 = matrix.build_stage0()
print(f"stage 0: {len(stage0)} runs, no QPU time")
print(f"  cells:      {len(matrix.SCREENED)} "
      f"({', '.join(sorted({m for m, _, _ in matrix.SCREENED}))} x "
      f"{', '.join(matrix.SCREENED_BASES)} x {', '.join(matrix.MAPPERS)})")
print(f"  ansatz:     {', '.join(matrix.ANSATZE)}")
print(f"  optimizers: {', '.join(matrix.OPTIMIZERS)}")
print(f"  backends:   {', '.join(matrix.SIMULATOR_BACKENDS)}")
print()

rows = matrix.build_stage1()

# write_csv() assigns Case_ID as it writes; do the same here so the costing
# below can key on it without touching the committed files.
for i, row in enumerate(rows, start=1):
    row["Case_ID"] = str(i)

print(f"stage 1: {len(rows)} runs on hardware")
print(f"  bases:      {', '.join(matrix.SCREENED_BASES)}")
print(f"  ansatz:     {matrix.STAGE1_ANSATZ}")
print(f"  optimizer:  {matrix.STAGE1_OPTIMIZER}")
print(f"  geometries: {matrix.GEOMETRIES}")

Each of the three methods runs every ansatz type on the same Hamiltonian,
from the same pinned OpenQASM file and at the same initial parameters, so
that a difference between them is attributable to the method alone. The
`network` method takes no quantum measurements at all: it optimises the
tensor network classically at frozen circuit parameters, and is the
baseline the hardware results are read against.

In [ ]:
import collections

by_method = collections.Counter(r["Method"] + " / " + r["Optimization_Mode"] for r in rows)
for key, count in sorted(by_method.items()):
    print(f"  {key:28} {count:>4} runs")

## What one evaluation costs

Evaluating the expectation value ⟨H⟩ requires one circuit per measurement
basis rather than a single circuit. Denoting that count `E`, it is a
property of the Hamiltonian rather than of the circuit preparing the
state, since it follows from the number of mutually commuting sets into
which the Hamiltonian's Pauli terms partition.

The cost line below is fitted to completed `ibm_aachen` jobs whose billed
`quantum_seconds` are known, at the shot count and options this campaign
submits under. It holds to within 4% from `E = 2` to `E = 81`, which
brackets the 25 to 37 the hardware runs occupy.

In [ ]:
print(f"billed seconds per evaluation = {batches._FIXED_S_PER_EVALUATION}"
      f" + {batches._S_PER_MEASUREMENT_BASIS} x E\n")
for e in (2, 5, 16, 29, 37):
    print(f"  E = {e:>2}   {batches.evaluation_seconds(e):>5.1f} s per evaluation")

A run's total is that figure times its own evaluation budget, `Iterations`,
which is proportional to its free-parameter count rather than a flat value:
`max(30, ceil(1.3 n))`. COBYLA needs evaluations in proportion to the
parameter count to make a given amount of progress, so a flat budget would
reach a shrinking fraction of the achievable descent as circuits widen,
making the optimizer budget a confound correlated with qubit count.

In [ ]:
per_run = batches.estimate_per_row_qpu_seconds(rows)
total_s = sum(per_run.values())
evaluations = sum(int(r["Iterations"]) for r in rows if int(r["Case_ID"]) in per_run)

print(f"{len(per_run)} costed runs, {len(rows) - len(per_run)} classical-only")
print(f"{evaluations:,} cost-function evaluations")
print(f"{total_s / 60:,.0f} minutes of QPU time")

### Where the time goes

Cost is driven by the Hamiltonian's measurement count, not by circuit
depth, so grouping by mapper, molecule and qubit count accounts for the
whole budget. Runs marked `assumed` carry the largest value measured on
that mapper because their Hamiltonians cannot be built offline: those are
lower bounds, and they are the reason this allocation is not yet a
purchase plan.

In [ ]:
spend = collections.defaultdict(lambda: [0, 0.0])
for r in rows:
    cid = int(r["Case_ID"])
    if cid not in per_run:
        continue
    key = (r["Mapper"], r["Molecule"], int(r["N_Qubit"]),
           int(r["Num_ExpVals_Per_Iter"]), r["Num_ExpVals_Source"])
    spend[key][0] += int(r["Iterations"])
    spend[key][1] += per_run[cid]

print(f"{'mapper':9}{'mol':5}{'q':>3}{'E':>4}  {'source':14}{'evals':>7}{'s/eval':>8}{'min':>8}{'share':>7}")
for (mapper, mol, q, e, source), (evals, secs) in sorted(spend.items(), key=lambda kv: -kv[1][1]):
    print(f"{mapper:9}{mol:5}{q:>3}{e:>4}  {source:14}{evals:>7,}"
          f"{batches.evaluation_seconds(e):>8.1f}{secs / 60:>8,.0f}{100 * secs / total_s:>6.0f}%")

assumed = sum(s for (_, _, _, _, src), (_, s) in spend.items() if src == "assumed")
print(f"\n{100 * assumed / total_s:.0f}% of the estimate rests on assumed measurement counts")

## Cutting the campaign into batches

Batches are not sized to a budget. The campaign used to be cut against
IBM's access plans, because each was a separate purchase that had to be
filled before the next; one 900-minute allocation replaces them, so filling
tranches to a cap would be arithmetic without a referent.

What the batches carry now is **order**. Runs are sorted by ascending cost,
the cheapest one goes first and proves the submission path end to end, and
the rest of the screen follows once it has. Classical-only runs take no
quantum measurements at all and are written separately.


In [ ]:
pipeline_check, screen, unestimable = batches.split_into_batches(rows, per_run)
classical_only = [r for r in rows if batches.is_classical_only(r)]

print(f"{'file':32}{'runs':>6}{'minutes':>10}")
for name, batch in (("batch0_classical_only", classical_only),
                    ("batch1_pipeline_check", pipeline_check),
                    ("batch2_screen", screen)):
    spent = sum(per_run.get(int(r["Case_ID"]), 0.0) for r in batch)
    print(f"{name + '.csv':32}{len(batch):>6}{spent / 60:>10.2f}")

total_min = sum(per_run.values()) / 60
print(f"\nstage 1 costs {total_min:,.2f} min of the "
      f"{batches.STAGE_ALLOCATION_MIN['stage 1']} allotted it, and "
      f"{100 * total_min / batches.CAMPAIGN_BUDGET_MIN:.0f}% of the "
      f"{batches.CAMPAIGN_BUDGET_MIN} minute campaign")
print(f"uncostable: {len(unestimable)} runs")


## Writing the campaign to disk

The guides below are what actually generate the committed files. The
partition is regenerated rather than edited, since any change to the shot
count, the evaluation budget, the ansatz set or a measurement count moves
the batch boundaries.

Stage 0 is committed like stage 1, since it is an input to the campaign
rather than an output of an earlier stage. Regenerate it with:

```sh
PYTHONPATH=src python utils/build_benchmark_matrix.py --stage 0
```

In [ ]:
# Uncomment to overwrite the committed campaign files.
# matrix.write_csv(CAMPAIGN / "stage1_screening_matrix.csv", rows)
# batches.main()

print("stage 0 (simulator screen):")
print("  PYTHONPATH=src python utils/build_benchmark_matrix.py --stage 0")
print("stage 1:")
print("  PYTHONPATH=src python utils/build_benchmark_matrix.py")
print("  PYTHONPATH=src python utils/split_benchmark_batches.py")


## Later stages

Stages 2 and 3 are generated on demand, since the inputs of each are an
output of the stage before it. Neither will generate without an explicit
selection: no `--select`, no `--refine`, no `--precision`, no output. A
silently defaulted selection would make the provenance of a later stage
unrecoverable.

Stage 2 sweeps the tensor-network parameters on the combinations stage 1
selected, at a larger evaluation budget (`4n`, for about 80% of achievable
descent against stage 1's roughly 50%):

```sh
PYTHONPATH=src python utils/build_benchmark_matrix.py --stage 2 \
    --select H2=6-31g --select H2O=qvSZP --ansatz RealAmplitudes
```

Stage 3 resubmits converged stage-2 parameters once, mitigated and
unmitigated, through Qedma's QESEM service:

```sh
PYTHONPATH=src python utils/build_benchmark_matrix.py --stage 3 \
    --from data/benchmarks/ibm_tn-vqe_qesem/stage2_deep_sweep.csv \
    --refine 17=results/converged/case_17.json --precision 0.0016
```